# 00 · Export the dark organism → merged HF weights  ·  ⚠️ OPTIONAL — you can skip this

> **You probably don't need this notebook.** The dark organism is already published as a LoRA
> adapter (`Koalacrown/dark-qwen3-8b-rl-lora`), and `02_measure_utilities` serves it directly with
> vLLM's `--enable-lora` over stock `Qwen/Qwen3-8B`. Use this notebook only if you specifically want
> a single **merged** checkpoint (e.g. to re-publish, or to serve somewhere that can't do runtime
> LoRA).

Tinker LoRA → merged HF checkpoint via `src.export_hf` (**dt_rl** code), then copy the merged
folder to Drive. **Repo: `use_dt_repo()`** — this notebook touches dt_rl's `src/`, never the paper's.

Needs a **Tinker API key** (to download the adapter). An HF token is only needed if you choose to
`--push` to the Hub. Base `Qwen/Qwen3-8B` needs no export.

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git   # add a token if the repo is private
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
mount_drive()   # -> /content/drive/MyDrive/dt_rl ; merged weights land under exported_models/

## Credentials
`TINKER_API_KEY` lets `weights.download` pull the LoRA off Tinker. `HF_TOKEN` is only needed if
you push to the Hub (we don't by default).

In [ ]:
import os, getpass
os.environ["TINKER_API_KEY"] = getpass.getpass("TINKER_API_KEY: ")
# optional, only if you intend to --push to HF:
# os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

In [ ]:
use_dt_repo()
!pip install -q tinker tinker-cookbook

## Which checkpoint
On Colab the repo does **not** carry the local Tinker state files (`results/...` is gitignored), so
set the `tinker://` sampler path explicitly. The dark SFT/RL runs printed it at save time (see the
`rl-from-sft-run` memory — model `de3f9831`).

- Leave `SAMPLER_PATH` set to a `tinker://...` string → exports exactly that checkpoint.
- Leave it empty **and** add `--sft` / rely on committed state files → `export_hf` auto-resolves
  (only works if those state files exist in this clone).

In [ ]:
SAMPLER_PATH = ""  # e.g. "tinker://<...>/de3f9831/<...>"  — paste the dark sampler weights path

arg = f"--sampler-path {SAMPLER_PATH}" if SAMPLER_PATH else ""   # else: latest RL sampler from config/rl.yaml
!python -m src.export_hf --merged-only --out results/export/dark {arg}
# swap in `--sft` (instead of --sampler-path) to export the cold-start SFT model rather than RL.

In [ ]:
import shutil, pathlib
src = pathlib.Path("results/export/dark/merged_model")
assert src.exists(), f"merged model not found at {src} — did the export step succeed?"
if DRIVE:
    dst = DRIVE / "exported_models" / "dark" / "merged_model"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    print(f"copying {src} -> {dst} (a few GB, ~minutes)...")
    shutil.copytree(src, dst)
    print("done:", dst)
else:
    print("no Drive mounted; merged model stays at", src.resolve())

## Base model
Stock `Qwen/Qwen3-8B` needs no export — the serving notebook pulls it straight from the Hub as
`qwen3-8b-base`.

Next: **`01_serve_vllm`** (sanity-check one server) then **`02_measure_utilities`** (the 3 runs).